# Read Parquet data

This notebook helps you load and inspect a Parquet file from this project.

Update the `parquet_path` variable below to point to your Parquet file.

In [ ]:
import pandas as pd

parquet_path = "data/clean/ilc_di12_gini.parquet"

df = pd.read_parquet(parquet_path)
print(len(df))
print(df.columns)

78
Index(['freq,age,statinfo,geo\TIME_PERIOD', '2014 ', '2015 ', '2016 ', '2017 ',
       '2018 ', '2019 ', '2020 ', '2021 ', '2022 ', '2023 ', '2024 ', '2025 '],
      dtype='str')


,"freq,age,statinfo,geo\TIME_PERIOD",2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,"A,TOTAL,GINI_HND,AL",:,:,:,36.8,35.4,34.3,33.2,33.0,31.0,30.2,:,:
1,"A,TOTAL,GINI_HND,AT",27.6,27.2,27.2,27.9,26.8,27.5,27.0,26.7,27.8,28.1,28.4,:
2,"A,TOTAL,GINI_HND,BE",25.9,26.2,26.3,26.1,25.7,25.1 b,25.3,24.1,24.7,24.3,24.6,23.4
3,"A,TOTAL,GINI_HND,BG",35.4,37.0,37.7 b,40.2,39.6,40.8,40.0,39.7,38.4,37.2,38.4,37.7
4,"A,TOTAL,GINI_HND,CH",29.5 b,29.6,29.4,30.1,29.7,30.6,31.2,31.4,31.0,31.5,31.0,:


In [17]:
import pandas as pd

# Load your Parquet file
df = pd.read_parquet("data/clean/ilc_di12_gini.parquet")

# Take the first column (string with comma-separated codes)
first_col = df.columns[0]

# Split into separate columns
split_cols = df[first_col].str.split(",", expand=True)

# Give them names (adjust as needed)
split_cols.columns = ["freq", "age", "unit", "geo"]  # or whatever labels you need

# Drop the original combined column and join the new ones
df = pd.concat([split_cols, df.drop(columns=[first_col])], axis=1)

df_de = df[df["geo"] == "DE"]

df_de = df_de[df_de["age"] == ("TOTAL")]

year_cols = [c for c in df_de.columns if c.strip().isdigit()]
for col in year_cols:
    df_de[col] = df_de[col].astype(str).str.extract(r"([\d.]+)")[0].astype(float)

df_de.head()

,freq,age,unit,geo,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
7,A,TOTAL,GINI_HND,DE,30.7,30.1,29.5,29.1,31.1,29.7,30.5,31.2,29.0,29.4,29.5,30.2


In [19]:
year_cols = [c for c in df_de.columns if c.strip().isdigit()]

df_long = df_de.melt(
    id_vars=["freq", "age", "unit", "geo"],
    value_vars=year_cols,
    var_name="year",
    value_name="value"
)

df_long["year"] = df_long["year"].astype(int)
df_long = df_long.dropna(subset=["value"])

In [ ]:
## always the same
df_long = df_long.drop(columns=["freq", "age"])

## renaming columns
df_long = df_long.rename(columns={"geo": "country", "unit": "indicator"})

## column order 
df_long = df_long[["country", "indicator", "year", "value"]]

df_long.to_parquet("data/processed/stg_gini.parquet", index=False)

,freq,age,unit,geo,year,value
0,A,TOTAL,GINI_HND,DE,2014,30.7
1,A,TOTAL,GINI_HND,DE,2015,30.1
2,A,TOTAL,GINI_HND,DE,2016,29.5
3,A,TOTAL,GINI_HND,DE,2017,29.1
4,A,TOTAL,GINI_HND,DE,2018,31.1
